# Stage 2 — Drift Trace Pipeline (Demo Notebook)

This notebook demonstrates the Stage 2 backward origin-tracing and forward spread-prediction interface using real OpenDrift simulations. The physics, uncertainty quantification, environmental-data fetching, and result-schema logic live in `ml/drift_trace/` — this notebook only calls into that module and visualizes the output, so the same tested code path is used here, in the backend, and in `ml/drift_trace/test_smoke.py`.

**Credentials**: real wind/current forcing requires `OPENWEATHER_API_KEY` and `COPERNICUSMARINE_SERVICE_USERNAME` / `COPERNICUSMARINE_SERVICE_PASSWORD` environment variables. Nothing in this repository hardcodes a key — see `ml/drift_trace/config.py`. If these are not set, the real-data cells below will raise a clear `MissingCredentialError`/`WindFetchError`/`CurrentsFetchError` instead of silently falling back to fabricated data.

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable as a package root (ml.drift_trace.*).
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from datetime import datetime, timedelta

from ml.drift_trace.trace import EnvironmentalForcing, trace_origin, simulate_forward
from ml.drift_trace.result import build_result_contract
from ml.drift_trace import wind as wind_module
from ml.drift_trace import currents as currents_module
from ml.drift_trace.config import EnvironmentCredentials

print("Stage 2 module ready")

## 1. Forward simulation (demo constants)

Predicts forward spread from a known spill point using explicit demo wind/current constants (no network calls) — useful for a quick sanity check of the physics/plotting path. Real forcing is used in the sections below.

In [ ]:
detection_time = datetime(2026, 8, 25, 0, 0)  # example: spill detection time
detection_lon, detection_lat = 51.5, 26.5  # central Persian Gulf, open water

demo_forcing = EnvironmentalForcing(
    wind_u=-5, wind_v=-2, current_u=-0.3, current_v=-0.1, sources=["demo:constant"]
)

forward_result = simulate_forward(
    detection_lon, detection_lat, detection_time, demo_forcing,
    duration_hours=24, particle_count=1000,
)
print("Forward Simulation Complete")
print(f"Particle stats: {forward_result.particle_stats}")
forward_result.simulation.plot(fast=True)

## 2. Backward trace (demo constants)

Traces the same detection point backward to estimate its origin. Uses `trace_origin()`'s physically-correct negative-`time_step` backward integration (see `ml/drift_trace/trace.py` module docstring for why this matters — negating environment vectors instead, as the original prototype did, keeps oil-weathering computations on and is not equivalent for real, non-constant reader fields).

In [ ]:
backward_result = trace_origin(
    detection_lon, detection_lat, detection_time, demo_forcing,
    trace_duration_hours=24, particle_count=1000,
)
print("Backward Simulation Complete")
backward_result.simulation.plot(fast=True)

## 3. Origin estimate with uncertainty (demo constants)

`trace_origin()` derives the origin centroid from each particle's own last *valid* position — not a blind mean over `isel(time=-1)`, which would silently drop every particle that stranded or otherwise terminated before the final timestep (the bug in the original prototype). It also reports 50%/95% dispersion radii, explicitly labeled as simulation-derived statistics rather than a statistical confidence interval.

In [ ]:
origin = backward_result.origin
stats = backward_result.particle_stats
print(f"Estimated origin: {origin['lon']:.4f} E, {origin['lat']:.4f} N at {origin['timestamp']}")
print(f"Uncertainty: 50% radius {origin['uncertainty_50_km']:.2f} km, "
      f"95% radius {origin['uncertainty_95_km']:.2f} km")
print(f"Stopped early: {origin['stopped_early']} (traced {origin['actual_hours_traced']:.2f}h)")
print(f"Particle outcomes: {stats}")

## 4. Real environmental forcing

Fetches real wind (OpenWeatherMap) and ocean currents (Copernicus Marine Service) for the detection point/time, caching each download locally under `ml/drift_trace/cache/` so repeated runs do not re-hit the rate-limited APIs. Raises immediately with a clear error if the required credentials are not set — this pipeline never substitutes fabricated wind/current data for a production trace.

In [ ]:
credentials = EnvironmentCredentials.from_env()

wind_u, wind_v = wind_module.get_wind_uv(detection_lat, detection_lon, credentials=credentials)
print(f"Wind u/v: {wind_u:.2f}, {wind_v:.2f} m/s")

currents_start = detection_time - timedelta(hours=48)
currents_end = detection_time + timedelta(hours=24)
currents_file = currents_module.fetch_currents(
    min_lon=48, max_lon=57, min_lat=24, max_lat=30,
    start=currents_start, end=currents_end,
    credentials=credentials,
)
print(f"Currents file: {currents_file}")

## 5. Backward trace with real forcing

Same `trace_origin()` call as above, but with the real fetched wind and current data instead of demo constants.

In [ ]:
real_forcing = EnvironmentalForcing(
    wind_u=wind_u, wind_v=wind_v, currents_file=currents_file,
    sources=["openweathermap:live", "copernicusmarine:live"],
)

real_trace_time = datetime(2026, 8, 27, 12, 0, 0)  # within the fetched currents window

real_backward_result = trace_origin(
    detection_lon, detection_lat, real_trace_time, real_forcing,
    trace_duration_hours=24, particle_count=1000,
)
print("Backward trace with real data complete")
real_backward_result.simulation.plot(fast=True)

## 6. Save Stage 2 result contract

Assembles the clean, JSON-serializable result contract (see `ml/drift_trace/result.py`) combining the backward trace and a forward re-simulation from the estimated origin, and writes it to `stage2_results.json` for the backend/frontend to consume.

In [ ]:
import json

real_forward_result = simulate_forward(
    real_backward_result.origin["lon"], real_backward_result.origin["lat"],
    real_trace_time, real_forcing,
    duration_hours=24, particle_count=1000,
)

contract = build_result_contract(real_backward_result, real_forward_result)

with open("stage2_results.json", "w") as f:
    json.dump(contract, f, indent=2)

print("Stage 2 results saved")
print(json.dumps(contract["origin"], indent=2))